# DAM2 Activity Analysis — reusable workflow

This notebook is a cleaned, reusable workflow for DAM2 locomotor activity data.

## Workflow
1. Configuration
2. Import and preparation of 5-min DAM2 data
3. Hourly activity
4. Total activity — before vs after
5. Total activity — before/after by Light/Dark
6. Hourly activity boxplots
7. Hourly activity profile — before vs after
8. Export for statistical analysis in R

### Important
- `before` = first 24 h, `after` = second 24 h.
- Light/Dark is derived from the DAM `LD` column (`1 = Light`, `0 = Dark`).
- Dead-fly detection is retained as a diagnostic only. Flies are **not automatically removed** unless you explicitly list them in the configuration.
- For a new fragrance/concentration, normally only edit the **CONFIGURATION** cell below.

In [ ]:
# ============================================================
# 1) IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from IPython.display import display


# ------------------------------------------------------------
# EDIT THESE SETTINGS FOR EACH NEW FRAGRANCE / CONCENTRATION
# ------------------------------------------------------------

FRAGRANCE_NAME = "F1"
CONCENTRATION_LABEL = "40µL"

# Change only if your input filenames use another naming scheme.
FILE_REP1 = f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}_Rep_1.csv"
FILE_REP2 = f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}_Rep_2.csv"

# Manually confirmed dead flies.
# Examples:
# DEAD_FLIES_REP1 = [3, 17]
# DEAD_FLIES_REP2 = [8]
# If none: None
DEAD_FLIES_REP1 = None
DEAD_FLIES_REP2 = [5, 6, 30]

REPLICATES = [
    {
        "replicate": "R1",
        "filepath": FILE_REP1,
        "dead_flies": DEAD_FLIES_REP1,
    },
    {
        "replicate": "R2",
        "filepath": FILE_REP2,
        "dead_flies": DEAD_FLIES_REP2,
    },
]


# ------------------------------------------------------------
# DAM2 SETTINGS
# ------------------------------------------------------------

BIN_LENGTH_MIN = 5
BINS_PER_HOUR = 12
MAX_HOURS = 48

CONDITION_ORDER = ["before", "after"]
PHASE_ORDER = ["Light", "Dark"]
REPLICATE_ORDER = [x["replicate"] for x in REPLICATES]


# ------------------------------------------------------------
# DEAD-FLY DIAGNOSTIC
# ------------------------------------------------------------

CHECK_DEAD_FLIES = True
ZERO_HOURS_THRESHOLD = 5

# Deliberately False:
# a long zero-activity period is only a warning and is not sufficient
# by itself to prove that a fly is dead.
AUTO_REMOVE_LIKELY_DEAD = False


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

# Separate folder for each fragrance and concentration.
CONCENTRATION_FOLDER = CONCENTRATION_LABEL.replace("µ", "u")

OUTPUT_DIR = (
    Path(f"activity_analysis_{FRAGRANCE_NAME}")
    / CONCENTRATION_FOLDER
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PREFIX = (
    f"{FRAGRANCE_NAME}_{CONCENTRATION_LABEL}"
    .replace("µ", "u")
)


# ------------------------------------------------------------
# PLOT STYLE
# ------------------------------------------------------------

condition_palette = {
    "before": "#fff7b2",
    "after": "#9ccfc8",
}

point_palette = {
    "before": "#9a8f35",
    "after": "#3f8f8b",
}

plt.style.use("default")
sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
})

print("Fragrance:", FRAGRANCE_NAME)
print("Concentration:", CONCENTRATION_LABEL)
print("Output folder:", OUTPUT_DIR.resolve())

## 2) Helper functions

The helper functions are defined only once and reused throughout the notebook.

The dead-fly check flags long zero-activity periods for inspection. It does **not** automatically exclude flies in the default configuration.

In [ ]:
# ============================================================
# 2) HELPER FUNCTIONS
# ============================================================

def add_phase_from_ld(data, ld_col="LD"):
    """Add Light/Dark phase from the DAM LD column."""
    data = data.copy()
    data[ld_col] = pd.to_numeric(data[ld_col], errors="coerce")

    phase = pd.Series(pd.NA, index=data.index, dtype="object")
    phase.loc[data[ld_col] == 1] = "Light"
    phase.loc[data[ld_col] == 0] = "Dark"

    data["Phase"] = pd.Categorical(
        phase,
        categories=PHASE_ORDER,
        ordered=True,
    )
    return data


def detect_potential_dead_flies_wide(
    df_wide,
    fly_cols,
    zero_hours_threshold=5,
    bins_per_hour=12,
):
    """
    Diagnostic dead-fly check.

    Flags:
    - any continuous zero-activity run >= threshold
    - trailing zero-activity run >= threshold at the end of recording
    """
    threshold_bins = zero_hours_threshold * bins_per_hour
    results = []

    for fly_col in fly_cols:
        activity_values = (
            pd.to_numeric(df_wide[fly_col], errors="coerce")
            .fillna(0)
            .to_numpy()
        )

        is_zero = activity_values == 0

        padded = np.concatenate([[False], is_zero, [False]])
        changes = np.diff(padded.astype(int))

        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]
        run_lengths = ends - starts

        if len(run_lengths) == 0:
            max_zero_bins = 0
            max_start = np.nan
            max_end = np.nan
        else:
            max_idx = np.argmax(run_lengths)
            max_zero_bins = int(run_lengths[max_idx])
            max_start = starts[max_idx]
            max_end = ends[max_idx] - 1

        if len(run_lengths) > 0 and ends[-1] == len(is_zero):
            trailing_zero_bins = int(run_lengths[-1])
            trailing_start = starts[-1]
            trailing_end = ends[-1] - 1
        else:
            trailing_zero_bins = 0
            trailing_start = np.nan
            trailing_end = np.nan

        def position_to_hour(position):
            if pd.isna(position):
                return np.nan
            return df_wide.iloc[int(position)]["Hour_raw"]

        results.append({
            "Fly": fly_col,
            "max_zero_bins": max_zero_bins,
            "max_zero_hours": max_zero_bins / bins_per_hour,
            "longest_zero_start_hour_raw": position_to_hour(max_start),
            "longest_zero_end_hour_raw": position_to_hour(max_end),
            "trailing_zero_bins": trailing_zero_bins,
            "trailing_zero_hours": trailing_zero_bins / bins_per_hour,
            "trailing_zero_start_hour_raw": position_to_hour(trailing_start),
            "trailing_zero_end_hour_raw": position_to_hour(trailing_end),
            "possible_dead_any_zero_block": (
                max_zero_bins >= threshold_bins
            ),
            "likely_dead_trailing_zero_block": (
                trailing_zero_bins >= threshold_bins
            ),
        })

    return pd.DataFrame(results)


def load_dam_activity_csv(
    filepath,
    replicate,
    dead_flies=None,
    check_dead_flies=True,
    zero_hours_threshold=5,
    auto_remove_likely_dead=False,
):
    """
    Load one DAM2 activity CSV.

    Output:
    one row = one fly × one 5-min bin.
    """
    data = pd.read_csv(filepath, sep=";", header=None)

    # Remove unused DAM columns.
    data = data.drop(
        columns=[3, 4, 5, 6, 7, 8],
        errors="ignore",
    )

    expected_cols = 4 + 32
    if data.shape[1] != expected_cols:
        raise ValueError(
            f"{filepath}: found {data.shape[1]} columns after cleaning; "
            f"expected {expected_cols}. Check the DAM export format."
        )

    data.columns = ["Index", "Date", "Time", "LD"] + [
        f"Fliege{i}" for i in range(1, 33)
    ]

    data["Index"] = pd.to_numeric(
        data["Index"],
        errors="coerce",
    )
    data["LD"] = pd.to_numeric(
        data["LD"],
        errors="coerce",
    )

    data["Replicate"] = replicate
    data["Hour_raw"] = (
        (data["Index"] - 1) // BINS_PER_HOUR
    )

    data = data[
        data["Hour_raw"] < MAX_HOURS
    ].copy()

    data["condition"] = np.where(
        data["Hour_raw"] < 24,
        "before",
        "after",
    )

    data["Hour"] = data["Hour_raw"] % 24

    fly_cols = [
        col for col in data.columns
        if col.startswith("Fliege")
    ]

    # --------------------------------------------------------
    # Dead-fly diagnostic BEFORE exclusion
    # --------------------------------------------------------
    if check_dead_flies:
        dead_check = detect_potential_dead_flies_wide(
            df_wide=data,
            fly_cols=fly_cols,
            zero_hours_threshold=zero_hours_threshold,
            bins_per_hour=BINS_PER_HOUR,
        )

        suspicious_any = dead_check[
            dead_check["possible_dead_any_zero_block"]
        ].copy()

        suspicious_trailing = dead_check[
            dead_check["likely_dead_trailing_zero_block"]
        ].copy()

        print("\n" + "=" * 60)
        print(f"Dead-fly check: {replicate}")
        print(
            f"Threshold: {zero_hours_threshold} h "
            "continuous zero activity"
        )
        print("=" * 60)

        print("\nPossible dead flies — any long zero block:")
        if suspicious_any.empty:
            print("None detected.")
        else:
            display(
                suspicious_any.sort_values(
                    "max_zero_hours",
                    ascending=False,
                )
            )

        print("\nLikely dead flies — long trailing zero block:")
        if suspicious_trailing.empty:
            print("None detected.")
        else:
            display(
                suspicious_trailing.sort_values(
                    "trailing_zero_hours",
                    ascending=False,
                )
            )

        if (
            auto_remove_likely_dead
            and not suspicious_trailing.empty
        ):
            auto_dead_columns = (
                suspicious_trailing["Fly"].tolist()
            )
            print(
                "\nAutomatically removing:",
                auto_dead_columns,
            )
            data = data.drop(
                columns=auto_dead_columns,
                errors="ignore",
            )

    # --------------------------------------------------------
    # Manual dead-fly exclusion
    # --------------------------------------------------------
    if dead_flies:
        dead_columns = [
            f"Fliege{i}" for i in dead_flies
        ]
        print(
            f"\nManually removing from {replicate}:",
            dead_columns,
        )
        data = data.drop(
            columns=dead_columns,
            errors="ignore",
        )

    fly_cols = [
        col for col in data.columns
        if col.startswith("Fliege")
    ]

    long = data.melt(
        id_vars=[
            "Index",
            "Date",
            "Time",
            "LD",
            "Replicate",
            "Hour_raw",
            "Hour",
            "condition",
        ],
        value_vars=fly_cols,
        var_name="Fly",
        value_name="activity",
    )

    long["activity"] = pd.to_numeric(
        long["activity"],
        errors="coerce",
    )

    long = long.dropna(
        subset=["activity"]
    ).copy()

    long["ID"] = (
        long["Replicate"].astype(str)
        + "_"
        + long["Fly"].astype(str)
    )

    long["condition"] = pd.Categorical(
        long["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    long["Replicate"] = pd.Categorical(
        long["Replicate"],
        categories=REPLICATE_ORDER,
        ordered=True,
    )

    long = add_phase_from_ld(
        long,
        ld_col="LD",
    )

    return long


def aggregate_hourly_activity(df_raw):
    """
    Aggregate 5-min activity counts to hourly activity.

    One row = one fly × one experimental hour.
    """
    hourly = (
        df_raw
        .groupby(
            [
                "ID",
                "Fly",
                "Replicate",
                "condition",
                "Hour_raw",
                "Hour",
            ],
            observed=True,
            as_index=False,
        )
        .agg(
            activity=("activity", "sum"),
            LD=("LD", "first"),
            bins_per_hour=("activity", "size"),
        )
    )

    hourly["condition"] = pd.Categorical(
        hourly["condition"],
        categories=CONDITION_ORDER,
        ordered=True,
    )

    hourly["Replicate"] = pd.Categorical(
        hourly["Replicate"],
        categories=REPLICATE_ORDER,
        ordered=True,
    )

    hourly = add_phase_from_ld(
        hourly,
        ld_col="LD",
    )

    return hourly


def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    print(f"Saved plot: {path}")


def add_mean_median_legend(ax):
    handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle="--",
            linewidth=1.2,
            label="Mean",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="-",
            linewidth=1.0,
            label="Median",
        ),
    ]

    ax.legend(
        handles=handles,
        frameon=True,
        facecolor="white",
        edgecolor="black",
    )


def plot_condition_box(
    data,
    y,
    ylabel,
    title,
    filename,
):
    fig, ax = plt.subplots(figsize=(7.5, 5))

    sns.boxplot(
        data=data,
        x="condition",
        y=y,
        order=CONDITION_ORDER,
        hue="condition",
        hue_order=CONDITION_ORDER,
        palette=condition_palette,
        dodge=False,
        showmeans=True,
        meanline=True,
        showfliers=False,
        medianprops={
            "color": "black",
            "linewidth": 1.0,
        },
        meanprops={
            "color": "black",
            "linewidth": 1.2,
            "linestyle": "--",
        },
        ax=ax,
    )

    sns.swarmplot(
        data=data,
        x="condition",
        y=y,
        order=CONDITION_ORDER,
        hue="condition",
        hue_order=CONDITION_ORDER,
        palette=point_palette,
        dodge=False,
        size=5,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
    )

    if ax.legend_ is not None:
        ax.legend_.remove()

    add_mean_median_legend(ax)

    ax.set_title(
        f"{FRAGRANCE_NAME}: {title}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Condition")
    ax.set_ylabel(ylabel)
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4,
    )

    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()


def plot_phase_box(
    data,
    y,
    ylabel,
    title,
    filename,
):
    fig, ax = plt.subplots(figsize=(8.5, 5.5))

    sns.boxplot(
        data=data,
        x="Phase",
        y=y,
        hue="condition",
        order=PHASE_ORDER,
        hue_order=CONDITION_ORDER,
        palette=condition_palette,
        showmeans=True,
        meanline=True,
        showfliers=False,
        medianprops={
            "color": "black",
            "linewidth": 1.0,
        },
        meanprops={
            "color": "black",
            "linewidth": 1.2,
            "linestyle": "--",
        },
        ax=ax,
    )

    sns.swarmplot(
        data=data,
        x="Phase",
        y=y,
        hue="condition",
        order=PHASE_ORDER,
        hue_order=CONDITION_ORDER,
        palette=point_palette,
        dodge=True,
        size=5,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.4,
        ax=ax,
    )

    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))

    condition_handles = [
        unique[c]
        for c in CONDITION_ORDER
        if c in unique
    ]

    condition_labels = [
        c
        for c in CONDITION_ORDER
        if c in unique
    ]

    mean_median_handles = [
        Line2D(
            [0], [0],
            color="black",
            linestyle="--",
            linewidth=1.2,
            label="Mean",
        ),
        Line2D(
            [0], [0],
            color="black",
            linestyle="-",
            linewidth=1.0,
            label="Median",
        ),
    ]

    ax.legend(
        condition_handles + mean_median_handles,
        condition_labels + ["Mean", "Median"],
        title="Condition",
        frameon=True,
        facecolor="white",
        edgecolor="black",
    )

    ax.set_title(
        f"{FRAGRANCE_NAME}: {title}",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("Phase")
    ax.set_ylabel(ylabel)
    ax.grid(
        axis="y",
        linestyle="--",
        alpha=0.4,
    )

    plt.tight_layout()
    save_figure(fig, filename)
    plt.show()

## 3) Load DAM2 data

Both replicates are loaded as 5-min data and combined into the central dataframe `df_raw`.

For a new dataset, the dead-fly lists are changed only in the configuration cell.

In [ ]:
# ============================================================
# 3) LOAD DATA
# ============================================================

replicate_frames = []

for config in REPLICATES:
    frame = load_dam_activity_csv(
        filepath=config["filepath"],
        replicate=config["replicate"],
        dead_flies=config["dead_flies"],
        check_dead_flies=CHECK_DEAD_FLIES,
        zero_hours_threshold=ZERO_HOURS_THRESHOLD,
        auto_remove_likely_dead=AUTO_REMOVE_LIKELY_DEAD,
    )
    replicate_frames.append(frame)

df_raw = pd.concat(
    replicate_frames,
    ignore_index=True,
)

df_raw = (
    df_raw
    .sort_values(
        ["Replicate", "Fly", "Index"]
    )
    .reset_index(drop=True)
)

df_raw["Fragrance"] = FRAGRANCE_NAME
df_raw["Concentration"] = CONCENTRATION_LABEL

print("\nRaw 5-min dataframe")
print("Shape:", df_raw.shape)
print("Number of flies:", df_raw["ID"].nunique())

print("\nObservations per fly:")
print(
    df_raw
    .groupby("ID", observed=True)
    .size()
    .value_counts()
    .sort_index()
)

print("\nMissing activity values:", df_raw["activity"].isna().sum())
print("Missing LD values:", df_raw["LD"].isna().sum())

display(df_raw.head())

## 4) Hourly activity

Twelve 5-min bins are summed to obtain one activity value per fly and hour.

`Hour_raw` runs from 0–47 across the full experiment, while `Hour` runs from 0–23 within each condition.

In [ ]:
# ============================================================
# 4) CREATE HOURLY ACTIVITY
# ============================================================

df_hourly = aggregate_hourly_activity(df_raw)

df_hourly["Fragrance"] = FRAGRANCE_NAME
df_hourly["Concentration"] = CONCENTRATION_LABEL

print("Hourly dataframe shape:", df_hourly.shape)
print("Number of flies:", df_hourly["ID"].nunique())

print("\nObservations per fly:")
print(
    df_hourly
    .groupby("ID", observed=True)
    .size()
    .value_counts()
    .sort_index()
)

print("\nHourly activity summary:")
print(df_hourly["activity"].describe())

display(df_hourly.head())

## 5) Total activity — before vs after

One row in `df_total_activity` represents one fly in one condition.

In [ ]:
# ============================================================
# 5) TOTAL ACTIVITY — BEFORE VS AFTER
# ============================================================

df_total_activity = (
    df_hourly
    .groupby(
        ["ID", "Replicate", "condition"],
        observed=True,
        as_index=False,
    )
    .agg(
        total_activity=("activity", "sum")
    )
)

print("Total activity per fly:")
display(df_total_activity.head())

print("\nSummary by condition:")
display(
    df_total_activity
    .groupby(
        "condition",
        observed=True,
    )
    .agg(
        n=("total_activity", "count"),
        mean=("total_activity", "mean"),
        sd=("total_activity", "std"),
        median=("total_activity", "median"),
        minimum=("total_activity", "min"),
        maximum=("total_activity", "max"),
    )
    .reset_index()
)

plot_condition_box(
    data=df_total_activity,
    y="total_activity",
    ylabel="Total activity [counts / 24 h]",
    title="Total Activity",
    filename=f"{OUTPUT_PREFIX}_total_activity.png",
)

## 6) Total activity — before/after by Light/Dark

This analysis is calculated directly from the **5-min data**, so the Light/Dark assignment comes from the original `LD` column rather than being inferred from clock time.

In [ ]:
# ============================================================
# 6) TOTAL ACTIVITY BY LIGHT / DARK
# ============================================================

df_activity_phase = (
    df_raw
    .dropna(subset=["Phase"])
    .groupby(
        [
            "ID",
            "Replicate",
            "condition",
            "Phase",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        total_activity=("activity", "sum")
    )
)

print("Activity by Light/Dark phase:")
display(df_activity_phase.head())

print("\nSummary:")
display(
    df_activity_phase
    .groupby(
        ["condition", "Phase"],
        observed=True,
    )
    .agg(
        n=("total_activity", "count"),
        mean=("total_activity", "mean"),
        sd=("total_activity", "std"),
        median=("total_activity", "median"),
        minimum=("total_activity", "min"),
        maximum=("total_activity", "max"),
    )
    .reset_index()
)

plot_phase_box(
    data=df_activity_phase,
    y="total_activity",
    ylabel="Total activity [counts]",
    title="Total Activity by Light/Dark Phase",
    filename=f"{OUTPUT_PREFIX}_total_activity_by_phase.png",
)

## 7) Hourly activity boxplots — before vs after

For each hour of the 24-h cycle, the distribution across individual flies is shown separately for `before` and `after`.

In [ ]:
# ============================================================
# 7) HOURLY ACTIVITY BOXPLOTS
# ============================================================

activity_hourly_plot = df_hourly.copy()

activity_hourly_plot["Hour"] = pd.to_numeric(
    activity_hourly_plot["Hour"],
    errors="coerce",
)

activity_hourly_plot = activity_hourly_plot.dropna(
    subset=["Hour", "activity", "condition"]
)

fig, ax = plt.subplots(figsize=(18, 7))

sns.boxplot(
    data=activity_hourly_plot,
    x="Hour",
    y="activity",
    hue="condition",
    order=list(range(24)),
    hue_order=CONDITION_ORDER,
    palette=condition_palette,
    showfliers=False,
    width=0.72,
    linewidth=1,
    ax=ax,
)

sns.stripplot(
    data=activity_hourly_plot,
    x="Hour",
    y="activity",
    hue="condition",
    order=list(range(24)),
    hue_order=CONDITION_ORDER,
    palette=point_palette,
    dodge=True,
    jitter=0.18,
    alpha=0.25,
    size=2,
    legend=False,
    ax=ax,
)

ax.set_title(
    f"{FRAGRANCE_NAME}: Hourly Activity Before and After Exposure",
    fontsize=15,
    fontweight="bold",
)

ax.set_xlabel(
    "Hour of the 24-h cycle",
    fontsize=12,
)

ax.set_ylabel(
    "Activity counts per hour",
    fontsize=12,
)

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))

ax.legend(
    [unique[c] for c in CONDITION_ORDER if c in unique],
    [c for c in CONDITION_ORDER if c in unique],
    title="Condition",
    frameon=True,
    facecolor="white",
    edgecolor="black",
)

sns.despine()
plt.tight_layout()

save_figure(
    fig,
    f"{OUTPUT_PREFIX}_hourly_activity_boxplots.png",
)

plt.show()

## 8) Hourly activity profile — before vs after

The hourly activity of each fly is first calculated individually.  
Then the mean activity across flies is plotted for every hour, with ±SE.

This is the **24-h activity profile / eduction-style plot**. It is not a frequency-domain periodogram. A true circadian periodogram is normally used to estimate rhythmic period from recordings spanning several days.

In [ ]:
# ============================================================
# 8) HOURLY ACTIVITY PROFILE — MEAN ± SE
# ============================================================

hourly_activity_summary = (
    df_hourly
    .groupby(
        ["condition", "Hour"],
        observed=True,
    )
    .agg(
        n=("activity", "count"),
        mean_activity=("activity", "mean"),
        sd_activity=("activity", "std"),
    )
    .reset_index()
)

hourly_activity_summary["se_activity"] = (
    hourly_activity_summary["sd_activity"]
    / np.sqrt(hourly_activity_summary["n"])
)

fig, ax = plt.subplots(figsize=(10, 5.5))

for condition in CONDITION_ORDER:
    condition_data = (
        hourly_activity_summary[
            hourly_activity_summary["condition"] == condition
        ]
        .sort_values("Hour")
    )

    x = condition_data["Hour"].to_numpy(dtype=float)
    y = condition_data["mean_activity"].to_numpy(dtype=float)
    se = condition_data["se_activity"].to_numpy(dtype=float)

    ax.plot(
        x,
        y,
        marker="o",
        markersize=4,
        linewidth=1.8,
        label=condition,
        color=point_palette[condition],
    )

    ax.fill_between(
        x,
        y - se,
        y + se,
        alpha=0.18,
        color=point_palette[condition],
    )

ax.set_title(
    f"{FRAGRANCE_NAME}: Hourly Activity Profile",
    fontsize=14,
    fontweight="bold",
)

ax.set_xlabel("Hour of the 24-h cycle")
ax.set_ylabel("Mean activity [counts / hour]")
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(0, 23)
ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.4,
)

ax.legend(
    title="Condition",
    frameon=True,
    facecolor="white",
    edgecolor="black",
)

plt.tight_layout()

save_figure(
    fig,
    f"{OUTPUT_PREFIX}_hourly_activity_profile.png",
)

plt.show()

display(hourly_activity_summary.head())

## 9) Build R dataframes

Four complementary export tables are created:

1. **Activity parameters** — one row per fly × condition  
   (`total_activity`, `total_activity_light`, `total_activity_dark`)

2. **Hourly activity** — one row per fly × condition × hour  
   for GLMMs including `Hour`

3. **Activity by phase** — long format with an explicit `Phase` column  
   (`Light` / `Dark`) for models comparing phases directly

4. **Activity parameters by phase** — same long-format phase table, exported again
   under a very explicit filename so it is easy to identify for R analyses


In [ ]:
# ============================================================
# 9) BUILD DATAFRAMES FOR R
# ============================================================

# ------------------------------------------------------------
# A) Main activity parameters:
# one row = one fly × condition
# ------------------------------------------------------------

phase_wide = (
    df_activity_phase
    .pivot(
        index=[
            "ID",
            "Replicate",
            "condition",
        ],
        columns="Phase",
        values="total_activity",
    )
    .rename(
        columns={
            "Light": "total_activity_light",
            "Dark": "total_activity_dark",
        }
    )
    .reset_index()
)

df_activity_parameters_R = (
    df_total_activity
    .merge(
        phase_wide,
        on=[
            "ID",
            "Replicate",
            "condition",
        ],
        how="left",
    )
)

df_activity_parameters_R.insert(
    0,
    "Fragrance",
    FRAGRANCE_NAME,
)

df_activity_parameters_R.insert(
    1,
    "Concentration",
    CONCENTRATION_LABEL,
)


# ------------------------------------------------------------
# B) Hourly activity for R-GLMM
# ------------------------------------------------------------

df_hourly_activity_R = df_hourly[
    [
        "ID",
        "Replicate",
        "condition",
        "Hour_raw",
        "Hour",
        "LD",
        "Phase",
        "activity",
    ]
].copy()

df_hourly_activity_R.insert(
    0,
    "Fragrance",
    FRAGRANCE_NAME,
)

df_hourly_activity_R.insert(
    1,
    "Concentration",
    CONCENTRATION_LABEL,
)


# ------------------------------------------------------------
# C) Activity by Light/Dark in long format
# ------------------------------------------------------------

df_activity_phase_R = df_activity_phase.copy()

df_activity_phase_R.insert(
    0,
    "Fragrance",
    FRAGRANCE_NAME,
)

df_activity_phase_R.insert(
    1,
    "Concentration",
    CONCENTRATION_LABEL,
)


# ------------------------------------------------------------
# Prepare text columns for CSV / R
# ------------------------------------------------------------

for dataframe in [
    df_activity_parameters_R,
    df_hourly_activity_R,
    df_activity_phase_R,
]:
    for col in [
        "Fragrance",
        "Concentration",
        "ID",
        "Replicate",
    ]:
        if col in dataframe.columns:
            dataframe[col] = (
                dataframe[col]
                .astype("string")
                .str.strip()
            )


# Sort
df_activity_parameters_R = (
    df_activity_parameters_R
    .sort_values(
        ["Replicate", "ID", "condition"]
    )
    .reset_index(drop=True)
)

df_hourly_activity_R = (
    df_hourly_activity_R
    .sort_values(
        ["Replicate", "ID", "condition", "Hour"]
    )
    .reset_index(drop=True)
)

df_activity_phase_R = (
    df_activity_phase_R
    .sort_values(
        ["Replicate", "ID", "condition", "Phase"]
    )
    .reset_index(drop=True)
)


print("Main activity-parameter dataframe:")
display(df_activity_parameters_R.head())

print("\nHourly R dataframe:")
display(df_hourly_activity_R.head())

print("\nLight/Dark long-format dataframe:")
display(df_activity_phase_R.head())

## 10) Export for R

All filenames are generated automatically from `FRAGRANCE_NAME` and `CONCENTRATION_LABEL`.

Existing files with the same names are overwritten when this cell is rerun.

**Important:**  
A single `Phase` column is only possible in a **long-format** export, because each fly has both a Light and a Dark value. Therefore, the notebook exports an additional long-format file with an explicit `Phase` column for direct modelling in R.


In [ ]:
# ============================================================
# 10) EXPORT FOR R
# ============================================================

R_PARAMETERS_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_activity_parameters_for_R.csv"
)

R_HOURLY_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_hourly_activity_for_R_GLMM.csv"
)

R_PHASE_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_activity_by_phase_for_R.csv"
)

R_PHASE_EXPLICIT_FILE = (
    OUTPUT_DIR
    / f"{OUTPUT_PREFIX}_activity_parameters_by_phase_for_R.csv"
)


df_activity_parameters_R.to_csv(
    R_PARAMETERS_FILE,
    index=False,
)

df_hourly_activity_R.to_csv(
    R_HOURLY_FILE,
    index=False,
)

df_activity_phase_R.to_csv(
    R_PHASE_FILE,
    index=False,
)

df_activity_phase_R.to_csv(
    R_PHASE_EXPLICIT_FILE,
    index=False,
)


print("\nExports completed:")
print(f"1. Main activity parameters:        {R_PARAMETERS_FILE.resolve()}")
print(f"2. Hourly activity:                 {R_HOURLY_FILE.resolve()}")
print(f"3. Activity by phase:               {R_PHASE_FILE.resolve()}")
print(f"4. Activity parameters by phase:    {R_PHASE_EXPLICIT_FILE.resolve()}")

## 11) Final checks

These checks help detect missing conditions, duplicate rows, or flies that do not have both `before` and `after` observations.

In [ ]:
# ============================================================
# 11) FINAL CHECKS
# ============================================================

print("Activity export checks")
print("----------------------")

print(
    "Number of flies:",
    df_activity_parameters_R["ID"].nunique(),
)

print("\nCondition counts:")
print(
    df_activity_parameters_R[
        "condition"
    ].value_counts(dropna=False)
)

duplicates = (
    df_activity_parameters_R
    .duplicated(
        subset=["ID", "condition"],
        keep=False,
    )
)

print(
    "\nDuplicated ID × condition rows:",
    int(duplicates.sum()),
)

if duplicates.any():
    display(
        df_activity_parameters_R.loc[
            duplicates
        ]
    )

observations_per_fly = (
    df_activity_parameters_R
    .groupby(
        "ID",
        observed=True,
    )
    .size()
)

print("\nObservations per fly:")
print(
    observations_per_fly
    .value_counts()
    .sort_index()
)

incomplete_flies = observations_per_fly[
    observations_per_fly != 2
]

print(
    "\nFlies without exactly two condition rows:",
    len(incomplete_flies),
)

if len(incomplete_flies) > 0:
    display(
        incomplete_flies
        .rename("n_observations")
        .reset_index()
    )

print("\nDone.")